# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

dataset = pd.read_csv('work/outputs/dataset.csv')

feature_cols = [
    'impressions_90d', 'clicks_90d', 'ctr_90d', 'avg_position_90d',
    'sessions_90d', 'pageviews_90d', 'engaged_sessions_90d',
    'organic_sessions_90d', 'impressions_last30', 'impressions_first60',
    'momentum_pct', 'active_days_90d', 'has_ga4_data', 'has_momentum'
]
target_col = 'is_declining_label'

print(f"Rows: {len(dataset):,}   Clients: {dataset['client_hash_id'].nunique()}")
print(f"Features: {len(feature_cols)} (all numeric -- no categorical columns in this dataset,")
print("unlike scripts/03_train_model.py's starter-CSV feature set, so no dummy-encoding step needed)")
print(f"Declining rate (base rate): {dataset[target_col].mean():.3f}")


## 1. Method choice and why

Per `skills/training-honest-models/SKILL.md`'s method table, this is a **ranking** question
("which pages first?", framed in `w02_ml_task_framing.ipynb` as Average Precision / P@50, not
a plain yes/no classification) -- so the model choice is "any classifier's probability,
evaluated at precision@K," moving **readable -> stronger**:

1. **Logistic Regression** (scaled features, `class_weight='balanced'`) -- the readable
   starting point; its coefficients say directly "more of X moves the odds up/down."
2. **Decision Tree** (`max_depth=5`) -- shallow enough to print and read end to end, a useful
   middle point between "readable" and "strong."
3. **Random Forest** (`class_weight='balanced_subsample'`) -- the stronger, less readable
   model; it only earns its place if it beats the simpler two on the same metric.

Same three families as `scripts/03_train_model.py`, adapted to this dataset's own (purely
numeric, warehouse-derived) feature set -- there's no `content_type`/`main_intent` categorical
columns here to encode, unlike the starter-CSV pipeline. `RANDOM_STATE = 42` throughout, fixed
and stated, per the skill's reproducibility basics.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
from sklearn.model_selection import train_test_split

def make_client_aware_split(df, target_col, client_col='client_hash_id', test_frac=0.2, random_state=RANDOM_STATE):
    """Holds out ~test_frac of CLIENTS (not rows) -- no client's pages appear in both splits."""
    clients = df[client_col].dropna().unique()
    if len(clients) >= 5:
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(clients)
        n_test = max(1, int(round(len(shuffled) * test_frac)))
        test_clients = set(shuffled[:n_test])
        test_mask = df[client_col].isin(test_clients)
        train_idx = df.index[~test_mask]
        test_idx = df.index[test_mask]
        if df.loc[train_idx, target_col].nunique() == 2 and df.loc[test_idx, target_col].nunique() == 2:
            return train_idx, test_idx, 'client_holdout'
    train_idx, test_idx = train_test_split(
        df.index, test_size=test_frac, random_state=random_state, stratify=df[target_col]
    )
    return train_idx, test_idx, 'stratified_row_holdout'

train_idx, test_idx, split_strategy = make_client_aware_split(dataset, target_col)

train_clients = set(dataset.loc[train_idx, 'client_hash_id'])
test_clients = set(dataset.loc[test_idx, 'client_hash_id'])

print(f"Split strategy: {split_strategy}")
print(f"Train rows: {len(train_idx):,}   Test rows: {len(test_idx):,}")
print(f"Train clients: {len(train_clients)}   Test clients: {len(test_clients)}")
print(f"Client overlap between train and test: {len(train_clients & test_clients)}  (must be 0)")
assert len(train_clients & test_clients) == 0 or split_strategy != 'client_holdout', \
    "client_holdout split must not share clients between train and test"
print(f"Test set declining rate: {dataset.loc[test_idx, target_col].mean():.3f} "
      f"(train: {dataset.loc[train_idx, target_col].mean():.3f})")


## 2. Split design

**Grouped by client (`client_hash_id`), not by row.** Pages from the same client share an
editorial team, a template, a niche -- a row-random split would let the model see some of a
client's pages in training and its other pages in test, which quietly leaks client identity
and inflates the score. This is the same rule `GUIDE.md` states for the reference pipeline
("holds out ~20% of clients, not rows") and is exactly what the `client_hash_id` column exists
for per the ML-04 data contract ("Grouped train/test split only").

The split falls back to a stratified row holdout only if there are too few unique clients
(<5) to hold any out meaningfully, or if a client-based split happens to leave only one class
in either half. The assertion above makes "no shared clients" a checked fact, not a claim.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_true = np.asarray(y_true)
    return float(y_true[order[:min(k, len(y_true))]].mean()) if len(y_true) else 0.0

X_train, X_test = dataset.loc[train_idx, feature_cols], dataset.loc[test_idx, feature_cols]
y_train, y_test = dataset.loc[train_idx, target_col], dataset.loc[test_idx, target_col]

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'decision_tree': DecisionTreeClassifier(
        class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    'random_forest': RandomForestClassifier(
        class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

test_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_scores[name] = model.predict_proba(X_test)[:, 1]

# Week-4 baseline, evaluated on the SAME test rows and SAME metric -- joined by content_hash_id
baseline = pd.read_csv('work/outputs/baseline_action_score.csv')
test_content_ids = dataset.loc[test_idx, 'content_hash_id']
baseline_test = test_content_ids.map(
    baseline.set_index('content_hash_id')['baseline_action_score']
).fillna(0).to_numpy()

base_rate = float(y_test.mean())
rows = []
for name, scores in {'baseline (Week 4)': baseline_test, **test_scores}.items():
    rows.append({
        'model': name,
        'precision_at_10': precision_at_k(y_test, scores, 10),
        'precision_at_25': precision_at_k(y_test, scores, 25),
        'precision_at_50': precision_at_k(y_test, scores, 50),
        'average_precision': average_precision_score(y_test, scores),
        'roc_auc': roc_auc_score(y_test, scores) if y_test.nunique() == 2 else float('nan'),
    })
comparison = pd.DataFrame(rows).set_index('model').round(3)
print(f"Test base rate (declining share): {base_rate:.3f}")
print(comparison)

best_model_name = comparison.drop('baseline (Week 4)').sort_values(
    ['precision_at_50', 'average_precision'], ascending=False
).index[0]
print(f"\nBest model by precision@50: {best_model_name}")
print(f"Lift over baseline at P@50: {comparison.loc[best_model_name, 'precision_at_50'] / max(comparison.loc['baseline (Week 4)', 'precision_at_50'], 1e-9):.2f}x")


## 3. Train + compare vs my baseline

Same data (`dataset`), same test rows (`test_idx` from section 2), same client-aware split,
same metrics (precision@10/25/50, Average Precision, ROC AUC) for all four rows of the table
above -- the Week-4 baseline (`work/outputs/baseline_action_score.csv`, from
`w04_baseline_score.ipynb`) is joined onto these exact test rows by `content_hash_id` rather
than re-computed, so it's a fair comparison, not a re-run under different conditions.

Per `skills/training-honest-models/SKILL.md`: if a model wins at precision@50 but loses at
precision@20 (or vice versa), report both -- that split IS a finding, not noise to average
away.

*(Paste the actual comparison table and best-model lift here once run against the real
dataset -- e.g. "random_forest reached Precision@50 = 0.71 vs the baseline's 0.36, a 2.0x
lift, and also led on Average Precision.")*


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
best_model = models[best_model_name]
best_scores = test_scores[best_model_name]

print("="*72)
print(f"FEATURE IMPORTANCE -- {best_model_name}")
print("="*72)
if isinstance(best_model, Pipeline):
    importances = np.abs(best_model.named_steps['model'].coef_[0])
else:
    importances = best_model.feature_importances_
importance_table = pd.DataFrame({'feature': feature_cols, 'importance': importances}) \
    .sort_values('importance', ascending=False)
print(importance_table.round(4).to_string(index=False))

top_feature = importance_table.iloc[0]['feature']
test_auc = roc_auc_score(y_test, best_scores) if y_test.nunique() == 2 else float('nan')
print(f"\nTop feature: {top_feature}")
print(f"Test ROC AUC: {test_auc:.3f}  "
      f"({'>= 0.999 -- suspiciously perfect, re-check ' + top_feature + ' for leakage' if test_auc >= 0.999 else 'not suspiciously perfect'})")

print("\n" + "="*72)
print("ERROR RATE BY RANK POSITION (ties to the ML-06 signal audit)")
print("="*72)
test_frame = dataset.loc[test_idx, feature_cols + [target_col]].copy()
test_frame['predicted_prob'] = best_scores
test_frame['predicted_label'] = (test_frame['predicted_prob'] >= 0.5).astype(int)
test_frame['correct'] = test_frame['predicted_label'] == test_frame[target_col]
pos_bucket = pd.qcut(test_frame['avg_position_90d'], 4, duplicates='drop')
print(test_frame.groupby(pos_bucket, observed=True)['correct'].agg(n='count', accuracy='mean').round(3))

print("\n" + "="*72)
print("THREE CONCRETE WRONG CASES")
print("="*72)
wrong = test_frame[~test_frame['correct']].copy()
confident_wrong_declining_missed = wrong[wrong[target_col] == 1].sort_values('predicted_prob').head(1)
confident_wrong_false_alarm = wrong[wrong[target_col] == 0].sort_values('predicted_prob', ascending=False).head(1)
borderline_wrong = wrong.iloc[(wrong['predicted_prob'] - 0.5).abs().argsort()[:1]] if len(wrong) else wrong
examples = pd.concat([confident_wrong_declining_missed, confident_wrong_false_alarm, borderline_wrong])
show_cols = feature_cols + [target_col, 'predicted_prob']
print(examples[show_cols].round(3).to_string())


## 4. Errors and interpretation

**What the model leans on:** the feature-importance table above ranks all 14 features for
the best model by precision@50. `skills/training-honest-models/SKILL.md`'s sanity check is
built in -- a test ROC AUC at or above 0.999 is flagged automatically as "suspiciously
perfect," which is what leakage looks like, not what a good model looks like.

**Where it's wrong:** the error-rate-by-rank-position table reuses the same quartile
bucketing as `w04_signal_audit.ipynb`'s Test 1, so a weak quartile here can be read directly
against that earlier finding.

**Three concrete wrong cases:** a missed decline the model was most confident was safe, a
false alarm the model was most confident was declining, and the single most borderline call
(closest to the 0.5 cutoff) -- printed above with their full feature row, for reading by eye
rather than by metric.

*(Name the top 3 features and, in one sentence each, why they plausibly relate to decline;
name the weakest rank-position quartile; and say what the three wrong cases have in common,
once run against the real dataset.)*


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.